In [ ]:
import subprocess
import neutralb1.utils as utils

WORKSPACE_DIR = utils.get_workspace_dir()

git_hash = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=WORKSPACE_DIR).decode('utf-8').strip()
print(git_hash)

# Fitting Final Results with a Sum of Breit-Wigner Distributions
Here I'll attempt to fit the mass-independent bins of the partial-wave analysis results with a simple model that is a sum of Breit-Wigners. Each section will explore a different model variation, and subsections will explore additional components like t-dependence or fixed parameters. The partial-wave results come from the final waveset [explored earlier](./final.ipynb)

Fit Details:
* Only spin-1 ($J=1$) waves
  * {$J^P\ell$} = {$1^+S, 1^-P, 1^+D$}
  * Waves all unconstrained, no D/S ratio or phase parameters
* No isotropic background term
* Uses all 4 polarization orientations
* Split data into bins of mass and $-t$
  * 20 MeV wide bins from $1.0 < M_{\omega\pi^0} < 1.8$ 
* Perform 500 randomized fits in each bin of mass and $-t$ independently
  * Additional 100 bootstrap samples in each bin to improve errors and study correlations


The basic idea behind each fit will be to use an intensity model like:
$$
I = \left|\sum_i c_i F_i(m, M_X, \Gamma_X, \ell)\right|^2
$$
where $c_i$ is a complex production coefficient paired with a relativistic Breit-Wigner $F_i$ that depends on the binned mass values $m$, the resonance mass $M_X$ and width $\Gamma_X$, and $\omega\pi$ angular momenta $\ell$. 

In [ ]:
# load common libraries
import pandas as pd
import pickle as pkl
import pathlib
import os, sys
import numpy as np
from uncertainties import ufloat, unumpy, umath
import matplotlib.pyplot as plt
import importlib.resources

# load neutralb1 libraries
import neutralb1.utils as utils
from neutralb1.analysis.result import ResultManager
import neutralb1.analysis.statistics as nb_stats

style_path = str(importlib.resources.files("neutralb1") / "neutralb1.mplstyle")
plt.style.use(style_path)

utils.load_environment()

# load in useful directories as constants
CWD = pathlib.Path.cwd()
STUDY_DIR = f"{WORKSPACE_DIR}/studies/data-fits/spin-1/no-iso-unconstrained/"

# set env variables for shell cells
os.environ["WORKSPACE_DIR"] = WORKSPACE_DIR
os.environ['STUDY_DIR'] = STUDY_DIR

In [ ]:
# load in the preprocessed results 
t_bins = ["t_0.10-0.16", "t_0.16-0.23", "t_0.23-0.35", "t_0.35-1.00"]
binned_results = {}
for t_bin in t_bins:
    with open(f"{STUDY_DIR}/{t_bin}/preprocessed_results_acceptance_corrected.pkl", "rb") as f:
        data = pkl.load(f)
        binned_results[t_bin] = ResultManager(**data)

binned_results: dict[str, ResultManager]

## Model Prerequisites
These definitions are useful for every individual model

In [ ]:
from neutralb1.analysis.physics import breit_wigner
vec_bw = np.vectorize(breit_wigner)
vec_circular_residual = np.vectorize(utils.circular_residual)

In [ ]:
def bw_phase_difference(bw1: complex, bw2: complex) -> float:
    """Calculate the phase difference (in degrees) between two Breit-Wigners

    Phases are wrapped from [-180, 180]

    Args:
        bw1 (complex): First Breit-Wigner
        bw2 (complex): Second Breit-Wigner
    """

    def wrap_phase(phase) -> float:
        return (phase + 180) % 360 - 180 # wrap to [-180, 180]


    phase1 = np.angle(bw1, deg=True)
    phase2 = np.angle(bw2, deg=True)
    diff = wrap_phase(phase1 - phase2)

    return diff

bw_phase_difference = np.vectorize(bw_phase_difference)

In [ ]:
import iminuit
import iminuit.cost


TODO: could build simple fit command that takes a model, a covariance matrix, other inputs, and simply performs the fit and bootstrap fits to return parameters + uncertainties

## Load Data

In [ ]:
# load in the preprocessed results 
t_bins = ["t_0.10-0.16", "t_0.16-0.23", "t_0.23-0.35", "t_0.35-1.00"]
binned_results = {}
for t_bin in t_bins:
    with open(f"{STUDY_DIR}/{t_bin}/preprocessed_results_acceptance_corrected.pkl", "rb") as f:
        data = pkl.load(f)
        binned_results[t_bin] = ResultManager(**data)

binned_results: dict[str, ResultManager]

## $b_1(1235)$ and $\rho'$ 
This section will explore when only a $b_1$ and $\rho'$ resonance is assumed to be present. We will fit the dominant waves in each -t bin: 
* $1^+S_0^{(+)}$
* $1^+D_0^{(+)}$
* $1^+S_{+1}^{(+)}$
* $1^-P_{+1}^{(+)}$

### Independent D/S ratio
Here we assume the D and S waves have independent scales

#### $-t$ independent

The below is currently a very simple setup for fitting the model, so the code is a bit sloppy and redundant at times

In [ ]:
def b1_rho_prime(
        mass: np.ndarray, 
        c_s0_real: float,
        c_s0_imag: float,
        c_s1_real: float,
        c_s1_imag: float,
        c_sm1_real: float,
        c_sm1_imag: float,
        c_d0_real: float,
        c_d0_imag: float,
        mass_b1: float, 
        width_b1: float, 
        c_p1_real: float,
        mass_rho_prime: float, 
        width_rho_prime: float) -> tuple[dict[str, float], dict[str, float]]:
    """
    BW model for the b1 and rho' (some rho(1450) like state) contributions to the amplitudes and phases

    Only natural waves will be fit.

    Args:
        mass: np.ndarray: The mass values to evaluate the model at
        c_s0_real: float: The real part of the complex coefficient for the S-wave m=0 amplitude
        c_s0_imag: float: The imaginary part of the complex coefficient for the S-wave m=0 amplitude
        c_s1_real: float: The real part of the complex coefficient for the S-wave m=+1 amplitude
        c_s1_imag: float: The imaginary part of the complex coefficient for the S-wave m=+1 amplitude
        c_sm1_real: float: The real part of the complex coefficient for the S-wave m=-1 amplitude
        c_sm1_imag: float: The imaginary part of the complex coefficient for the S-wave m=-1 amplitude
        c_d0_real: float: The real part of the complex coefficient for the D-wave m=0 amplitude
        c_d0_imag: float: The imaginary part of the complex coefficient for the D-wave m=0 amplitude
        mass_b1: float: The mass of the b1 resonance
        width_b1: float: The width of the b1 resonance
        c_p1_real: float: The real part of the complex coefficient for the P-wave m=+1 amplitude
        mass_rho_prime: float: The mass of the rho' resonance
        width_rho_prime: float: The width of the rho' resonance

    Returns:
        tuple[dict[str, float], dict[str, float]]: A tuple containing two dictionaries:
            - The first dictionary contains the intensities of the S-wave, D-wave, and P-wave amplitudes.
            - The second dictionary contains the phase differences between the S-wave, D-wave, and P-wave amplitudes.
    """
    c_s0 = complex(c_s0_real, c_s0_imag)
    c_s1 = complex(c_s1_real, c_s1_imag)
    c_sm1 = complex(c_sm1_real, c_sm1_imag)
    c_d0 = complex(c_d0_real, c_d0_imag)
    c_p1 = complex(c_p1_real, 0.0) # P-wave is fixed to real to resolve unconstrained phase ambiguity

    s0_bw = c_s0 * vec_bw(mass, mass_b1, width_b1, 0)
    s1_bw = c_s1 * vec_bw(mass, mass_b1, width_b1, 0)
    sm1_bw = c_sm1 * vec_bw(mass, mass_b1, width_b1, 0)
    d0_bw = c_d0 * vec_bw(mass, mass_b1, width_b1, 2)
    p1_bw = c_p1 * vec_bw(mass, mass_rho_prime, width_rho_prime, 1)

    intensity_s0 = np.abs(s0_bw)**2
    intensity_s1 = np.abs(s1_bw)**2
    intensity_sm1 = np.abs(sm1_bw)**2
    intensity_d0 = np.abs(d0_bw)**2
    intensity_p1 = np.abs(p1_bw)**2

    phase_s0_s1 = bw_phase_difference(s0_bw, s1_bw)
    phase_s0_sm1 = bw_phase_difference(s0_bw, sm1_bw)
    phase_s0_d0 = bw_phase_difference(s0_bw, d0_bw)
    phase_s0_p1 = bw_phase_difference(s0_bw, p1_bw)
    phase_s1_p1 = bw_phase_difference(s1_bw, p1_bw)
    phase_s1_sm1 = bw_phase_difference(s1_bw, sm1_bw)
    phase_s1_d0 = bw_phase_difference(s1_bw, d0_bw)
    phase_sm1_p1 = bw_phase_difference(sm1_bw, p1_bw)
    phase_sm1_d0 = bw_phase_difference(sm1_bw, d0_bw)
    phase_d0_p1 = bw_phase_difference(d0_bw, p1_bw)

    intensities = {
        "intensity_s0": intensity_s0,
        "intensity_s1": intensity_s1,
        "intensity_sm1": intensity_sm1,
        "intensity_d0": intensity_d0,
        "intensity_p1": intensity_p1
    }

    phases = {
        "phase_s0_s1": phase_s0_s1,
        "phase_s0_sm1": phase_s0_sm1,
        "phase_s0_d0": phase_s0_d0,
        "phase_s0_p1": phase_s0_p1,
        "phase_s1_p1": phase_s1_p1,
        "phase_s1_sm1": phase_s1_sm1,
        "phase_s1_d0": phase_s1_d0,
        "phase_sm1_p1": phase_sm1_p1,
        "phase_sm1_d0": phase_sm1_d0,
        "phase_d0_p1": phase_d0_p1,
    }

    return (intensities, phases)
    


In [ ]:
def residuals(
    params,
    mass: np.ndarray,
    intensities: dict[str, tuple[np.ndarray, np.ndarray]],
    phases: dict[str, tuple[np.ndarray, np.ndarray]],
) -> np.ndarray:
    """
    Calculate the residuals between the model and the data for both intensities and phases.

    Args:
        params: The parameters for the model.
        mass: The mass values to evaluate the model at.
        intensities: The observed intensities and their uncertainties for each wave.
        phases: The observed phase differences and their uncertainties for each wave pair.
    """
    predicted_intensities, predicted_phases = b1_rho_prime(
        mass,
        *params
    )

    # Calculate the residuals for intensities
    intensity_residuals = []
    for key, (i_obs, i_err) in intensities.items():
        i_pred = predicted_intensities[key]
        intensity_residuals.append((i_pred - i_obs) / i_err)

    # Calculate the residuals for phases
    phase_residuals = []
    for key, (p_obs, p_err) in phases.items():
        p_pred = predicted_phases[key]
        r_plus = vec_circular_residual(p_pred, p_obs, in_degrees=True) / p_err
        r_minus = vec_circular_residual(p_pred, -p_obs, in_degrees=True) / p_err
        r = np.where(np.abs(r_plus) < np.abs(r_minus), r_plus, r_minus)

        phase_residuals.append(r)

    # Combine the residuals
    return np.concatenate(intensity_residuals + phase_residuals)

In [ ]:
from scipy.optimize import least_squares

In [ ]:
# Store b1, rho fit parameters and uncertainties for each t bin globally
b1_rho_prime_fit_results = {}

In [ ]:
from matplotlib.ticker import ScalarFormatter, MaxNLocator

colors = plt.colormaps["Dark2"].colors # type: ignore

for bin_idx, (t_bin, results) in enumerate(binned_results.items()):

    # Get the amplitude and phase data values
    s0_name, s1_name, sm1_name, p1_name, d0_name = "p1p0S", "p1ppS", "p1pmS", "p1mpP", "p1p0D"
    s0_s1_name = results.phase_difference_dict[(s0_name, s1_name)]
    s0_d0_name = results.phase_difference_dict[(s0_name, d0_name)]
    s0_p1_name = results.phase_difference_dict[(s0_name, p1_name)]
    s0_sm1_name = results.phase_difference_dict[(s0_name, sm1_name)]
    s1_p1_name = results.phase_difference_dict[(s1_name, p1_name)]
    s1_d0_name = results.phase_difference_dict[(s1_name, d0_name)]
    s1_sm1_name = results.phase_difference_dict[(s1_name, sm1_name)]
    sm1_p1_name = results.phase_difference_dict[(sm1_name, p1_name)]
    sm1_d0_name = results.phase_difference_dict[(sm1_name, d0_name)]
    d0_p1_name = results.phase_difference_dict[(d0_name, p1_name)]

    intensity_s0 = results.fit_df[s0_name].to_numpy()
    intensity_s1 = results.fit_df[s1_name].to_numpy()
    intensity_sm1 = results.fit_df[sm1_name].to_numpy()
    intensity_p1 = results.fit_df[p1_name].to_numpy()
    intensity_d0 = results.fit_df[d0_name].to_numpy()

    intensity_s0_err = results.plot.intensity.get_bootstrap_error(s0_name).to_numpy()
    intensity_s1_err = results.plot.intensity.get_bootstrap_error(s1_name).to_numpy()
    intensity_sm1_err = results.plot.intensity.get_bootstrap_error(sm1_name).to_numpy()
    intensity_p1_err = results.plot.intensity.get_bootstrap_error(p1_name).to_numpy()
    intensity_d0_err = results.plot.intensity.get_bootstrap_error(d0_name).to_numpy()

    phase_s0_s1 = results.fit_df[s0_s1_name].to_numpy()
    phase_s0_d0 = results.fit_df[s0_d0_name].to_numpy()
    phase_s0_p1 = results.fit_df[s0_p1_name].to_numpy()
    phase_s0_sm1 = results.fit_df[s0_sm1_name].to_numpy()
    phase_s1_p1 = results.fit_df[s1_p1_name].to_numpy()
    phase_s1_d0 = results.fit_df[s1_d0_name].to_numpy()
    phase_s1_sm1 = results.fit_df[s1_sm1_name].to_numpy()
    phase_sm1_p1 = results.fit_df[sm1_p1_name].to_numpy()
    phase_sm1_d0 = results.fit_df[sm1_d0_name].to_numpy()
    phase_d0_p1 = results.fit_df[d0_p1_name].to_numpy()

    phase_s0_s1_err = results.plot.phase.get_bootstrap_error(s0_s1_name).to_numpy()
    phase_s0_d0_err = results.plot.phase.get_bootstrap_error(s0_d0_name).to_numpy()
    phase_s0_p1_err = results.plot.phase.get_bootstrap_error(s0_p1_name).to_numpy()
    phase_s0_sm1_err = results.plot.phase.get_bootstrap_error(s0_sm1_name).to_numpy()
    phase_s1_p1_err = results.plot.phase.get_bootstrap_error(s1_p1_name).to_numpy()
    phase_s1_d0_err = results.plot.phase.get_bootstrap_error(s1_d0_name).to_numpy()
    phase_s1_sm1_err = results.plot.phase.get_bootstrap_error(s1_sm1_name).to_numpy()
    phase_sm1_p1_err = results.plot.phase.get_bootstrap_error(sm1_p1_name).to_numpy()
    phase_sm1_d0_err = results.plot.phase.get_bootstrap_error(sm1_d0_name).to_numpy()
    phase_d0_p1_err = results.plot.phase.get_bootstrap_error(d0_p1_name).to_numpy()

    data_masses = np.array(results.get_mass_centers())
    bin_width = results.get_mass_bin_width()


    # get some initial guesses for the fit parameters
    c_s0_mag = np.sqrt(np.mean(intensity_s0))
    c_s1_mag = np.sqrt(np.mean(intensity_s1))
    c_sm1_mag = np.sqrt(np.mean(intensity_sm1))
    c_d0_mag = np.sqrt(np.mean(intensity_d0))
    c_p1_mag = np.sqrt(np.mean(intensity_p1))
    
    c_s0_guess = c_s0_mag * np.exp(1j * 0)
    c_s1_guess = c_s1_mag * np.exp(1j * 0)
    c_sm1_guess = c_sm1_mag * np.exp(1j * 0)
    c_d0_guess = c_d0_mag * np.exp(1j * 0)
    c_p1_guess = c_p1_mag * np.exp(1j * 0)

    mass_b1_guess = 1.235
    mass_rho_prime_guess = 1.465
    width_b1_guess = 0.142
    width_rho_prime_guess = 0.400

    # perform the fit with iterative phase-branch selection.

    popt = np.array([
        c_s0_guess.real,
        c_s0_guess.imag,
        c_s1_guess.real,
        c_s1_guess.imag,
        c_sm1_guess.real,
        c_sm1_guess.imag,
        c_d0_guess.real,
        c_d0_guess.imag,
        mass_b1_guess,
        width_b1_guess,
        c_p1_guess.real,
        mass_rho_prime_guess,
        width_rho_prime_guess,
    ], dtype=float)

    fit = least_squares(
        residuals,
        x0=popt,
        args=(
            data_masses,
            {
                "intensity_s0": (intensity_s0, intensity_s0_err),
                "intensity_s1": (intensity_s1, intensity_s1_err),
                "intensity_sm1": (intensity_sm1, intensity_sm1_err),
                "intensity_d0": (intensity_d0, intensity_d0_err),
                "intensity_p1": (intensity_p1, intensity_p1_err),
            },
            {
                "phase_s0_s1": (phase_s0_s1, phase_s0_s1_err),
                "phase_s0_d0": (phase_s0_d0, phase_s0_d0_err),
                "phase_s0_p1": (phase_s0_p1, phase_s0_p1_err),
                "phase_s0_sm1": (phase_s0_sm1, phase_s0_sm1_err),
                "phase_s1_p1": (phase_s1_p1, phase_s1_p1_err),
                "phase_s1_d0": (phase_s1_d0, phase_s1_d0_err),
                "phase_s1_sm1": (phase_s1_sm1, phase_s1_sm1_err),
                "phase_sm1_p1": (phase_sm1_p1, phase_sm1_p1_err),
                "phase_sm1_d0": (phase_sm1_d0, phase_sm1_d0_err),
                "phase_d0_p1": (phase_d0_p1, phase_d0_p1_err),
            },
        ),
        bounds=(
            [
                -np.inf, -np.inf, -np.inf, -np.inf, -np.inf, -np.inf, -np.inf, -np.inf, 
                1.2, 0.1, -np.inf, 1.2, 0.2
            ],
            [
                np.inf, np.inf, np.inf, np.inf, np.inf, np.inf, np.inf, np.inf,
                1.4, 0.4, np.inf, 1.7, 0.7
            ]
        )
    )

    # Extract the fit parameters and their uncertainties
    popt = fit.x
    mass_b1_fit, width_b1_fit = popt[8], popt[9]
    mass_rho_prime_fit, width_rho_prime_fit = popt[11], popt[12]

    J = fit.jac
    cov = np.linalg.inv(J.T @ J)
    dof = len(fit.fun) - len(popt)
    s_sq = 2 * fit.cost / dof
    cov = cov * s_sq
    perr = np.sqrt(np.diag(cov))

    mass_b1_fit_err, width_b1_fit_err = perr[8], perr[9]
    mass_rho_prime_fit_err, width_rho_prime_fit_err = perr[11], perr[12]

    b1_rho_prime_fit_results[t_bin] = {
        "mass_b1": mass_b1_fit,
        "width_b1": width_b1_fit,
        "mass_rho_prime": mass_rho_prime_fit,
        "width_rho_prime": width_rho_prime_fit,
        "mass_b1_err": mass_b1_fit_err,
        "width_b1_err": width_b1_fit_err,
        "mass_rho_prime_err": mass_rho_prime_fit_err,
        "width_rho_prime_err": width_rho_prime_fit_err
    }

    # Create a 4x4 grid of subplots for the amplitudes and phases
    fig, axs = plt.subplots(
        5, 5,
        layout="constrained",
        dpi=300
    )

    grid_amplitudes = [
        [sm1_name, s0_sm1_name, s1_sm1_name, sm1_p1_name, sm1_d0_name],
        [s0_name, s0_s1_name, s0_p1_name, s0_d0_name],
        [s1_name, s1_p1_name, s1_d0_name],
        [p1_name, d0_p1_name],
        [d0_name]
    ]

    translate_dict = {
        s0_name: "intensity_s0",
        s1_name: "intensity_s1",
        sm1_name: "intensity_sm1",
        p1_name: "intensity_p1",
        d0_name: "intensity_d0",
        s0_s1_name: "phase_s0_s1",
        s0_d0_name: "phase_s0_d0",
        s0_p1_name: "phase_s0_p1",
        s0_sm1_name: "phase_s0_sm1",
        s1_p1_name: "phase_s1_p1",
        s1_d0_name: "phase_s1_d0",
        s1_sm1_name: "phase_s1_sm1",
        sm1_p1_name: "phase_sm1_p1",
        sm1_d0_name: "phase_sm1_d0",
        d0_p1_name: "phase_d0_p1"
    }

    fine_masses = np.linspace(1.0, 1.8, 1000)
    fit_intensities, fit_phases = b1_rho_prime(fine_masses, *popt)

    for row in range(5):
        for col in range(5):
            if col < row: # skip lower triangle, redundant information
                axs[row, col].axis('off')
                continue
        
            amp_name = grid_amplitudes[row][col-row]
            y_values = results.fit_df[amp_name].to_numpy()
            y_errors = results.plot.intensity.get_bootstrap_error(amp_name).to_numpy()

            if row == col:
                # ==== PLOT INTENSITIES ====                

                # plot original data with error bars
                color = colors[2] if "S" in amp_name or "D" in amp_name else colors[3]                
                axs[row, col].errorbar(
                    x=data_masses,
                    xerr=bin_width/2,
                    y=y_values,
                    yerr=y_errors,
                    color=color,
                    marker=".",
                    markersize=4,
                    linestyle="",
                    alpha=0.7
                )
                
                # plot the fitted model
                axs[row, col].plot(
                    fine_masses,
                    fit_intensities[translate_dict[amp_name]],
                    color=color,
                    marker="",
                    linestyle="-",
                    linewidth=1.5,                    
                )

                axs[row, col].set_ylim(bottom=0)
                axs[row, col].yaxis.set_major_locator(MaxNLocator(nbins=5, integer=True))
                axs[row, col].ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
                axs[row, col].yaxis.set_major_formatter(ScalarFormatter(useMathText=True))
                axs[row, col].yaxis.get_major_formatter().set_powerlimits((0, 0))
                axs[row, col].yaxis.set_offset_position("left")
                axs[row, col].yaxis.get_offset_text().set_fontsize(8)
                axs[row, col].tick_params(axis="y", labelsize=8)
                axs[row, col].set_xlabel(r"$\omega\pi^0$ inv. mass (GeV)", fontsize=8)
                axs[row, col].set_ylabel(f"Events / {bin_width:.3f} GeV", loc="top", fontsize=8)


            else:
                # ==== PLOT PHASES ====

                # plot original data with error bars
                axs[row,col].errorbar(
                    x=data_masses,
                    xerr=bin_width/2,
                    y=y_values,
                    yerr=y_errors,
                    color="black",
                    marker=".",
                    markersize=4,
                    linestyle="",
                    alpha=0.7
                )
                axs[row,col].errorbar(
                    x=data_masses,
                    xerr=bin_width/2,
                    y=-y_values, # sign ambiguous phase 
                    yerr=y_errors,
                    color="black",
                    marker=".",
                    markersize=4,
                    linestyle="",
                    alpha=0.7
                )
                axs[row,col].set_ylim(-180, 180)
                axs[row,col].set_yticks(np.linspace(-180, 180, 5))
                axs[row,col].set_yticklabels([f"{x:.0f}" for x in np.linspace(-180, 180, 5)], fontsize=8)

                # plot the fitted model
                axs[row, col].plot(
                    fine_masses,
                    fit_phases[translate_dict[amp_name]],
                    color="black",
                    marker="",
                    linestyle="-",
                    linewidth=1.5,                    
                )

            
            axs[row, col].set_xticks(np.linspace(1.0, 1.8, 6))
            axs[row, col].set_xticklabels([f"{x:.1f}" for x in np.linspace(1.0, 1.8, 6)], fontsize=8)
            latex_amp_name = utils.convert_amp_name(amp_name)
            axs[row, col].set_title(latex_amp_name, loc="right", fontsize=10)

            

    t_low, t_high = results.get_t_edges()
    fig.suptitle(f"{t_low:.2f} < t < {t_high:.2f} GeV^2", ha="center", fontsize=16)

    ds0_ratio = np.sqrt(fit_intensities["intensity_d0"] / fit_intensities["intensity_s0"])
    e852_mass_range = [1.155, 1.315]
    filtered_ds0_ratio = ds0_ratio[(fine_masses >= e852_mass_range[0]) & (fine_masses <= e852_mass_range[1])]
    mean_ds0_ratio = np.mean(filtered_ds0_ratio)
    

    fig.text(0.25, 0.15, 
        (
            "Fit results:"
            "\n"
            rf"$M_{{b1}} = {mass_b1_fit:.3f} \pm {mass_b1_fit_err:.3f}$ GeV"
            "\n"
            rf"$\Gamma_{{b1}} = {width_b1_fit:.3f} \pm {width_b1_fit_err:.3f}$ GeV"
            "\n\n"
            rf"$M_{{\rho'}} = {mass_rho_prime_fit:.3f} \pm {mass_rho_prime_fit_err:.3f}$ GeV"
            "\n"
            rf"$\Gamma_{{\rho'}} = {width_rho_prime_fit:.3f} \pm {width_rho_prime_fit_err:.3f}$ GeV"
            "\n\n"
            rf"$\overline{{D/S}}_0 = {mean_ds0_ratio:.3f} \pm {np.std(filtered_ds0_ratio):.3f}$"
        ),
        bbox=dict(facecolor='white', alpha=0.5, edgecolor='black'),
        ha="center", fontsize=14
    )

    plt.show()



#### $-t$ dependent

## $b_(1235)$, $\rho'$, + Background
Try adding a non-resonant real-valued polynomial background term to see if it absorbs some of the intensity.

### Independent D/S Ratio

#### -t independent

In [ ]:
def bkg_polynomial(mass: np.ndarray, a0: float, a1: float, a2: float, a3: float, a4: float) -> np.ndarray:
    """
    Evaluate the background polynomial at the given mass values.

    Args:
        mass: np.ndarray: The mass values to evaluate the polynomial at
        a0: float: The constant background parameter
        a1: float: The linear background parameter
        a2: float: The quadratic background parameter
        a3: float: The cubic background parameter
        a4: float: The quartic background parameter

    Returns:
        np.ndarray: The evaluated polynomial values at the given mass values
    """
    return a0 + a1 * mass + a2 * mass**2 + a3 * mass**3 + a4 * mass**4

In [ ]:
def b1_rho_prime_bkg(
        mass: np.ndarray, 
        c_s0_real: float,
        c_s0_imag: float,
        c_s1_real: float,
        c_s1_imag: float,
        c_sm1_real: float,
        c_sm1_imag: float,
        c_d0_real: float,
        c_d0_imag: float,
        mass_b1: float, 
        width_b1: float, 
        c_p1_real: float,
        mass_rho_prime: float, 
        width_rho_prime: float,
        a0: float,
        a1: float,
        a2: float,
        a3: float,
        a4: float

    ) -> tuple[dict[str, float], dict[str, float]]:
    """
    BW model for the b1 and rho' (some rho(1450) like state) contributions to the amplitudes and phases

    Only natural waves will be fit.

    Args:
        mass: np.ndarray: The mass values to evaluate the model at
        c_s0_real: float: The real part of the complex coefficient for the S-wave m=0 amplitude
        c_s0_imag: float: The imaginary part of the complex coefficient for the S-wave m=0 amplitude
        c_s1_real: float: The real part of the complex coefficient for the S-wave m=+1 amplitude
        c_s1_imag: float: The imaginary part of the complex coefficient for the S-wave m=+1 amplitude
        c_sm1_real: float: The real part of the complex coefficient for the S-wave m=-1 amplitude
        c_sm1_imag: float: The imaginary part of the complex coefficient for the S-wave m=-1 amplitude
        c_d0_real: float: The real part of the complex coefficient for the D-wave m=0 amplitude
        c_d0_imag: float: The imaginary part of the complex coefficient for the D-wave m=0 amplitude
        mass_b1: float: The mass of the b1 resonance
        width_b1: float: The width of the b1 resonance
        c_p1_real: float: The real part of the complex coefficient for the P-wave m=+1 amplitude
        mass_rho_prime: float: The mass of the rho' resonance
        width_rho_prime: float: The width of the rho' resonance
        a0: float: The constant background parameter
        a1: float: The linear background parameter
        a2: float: The quadratic background parameter
        a3: float: The cubic background parameter
        a4: float: The quartic background parameter

    Returns:
        tuple[dict[str, float], dict[str, float]]: A tuple containing two dictionaries:
            - The first dictionary contains the intensities of the S-wave, D-wave, P-wave amplitudes, and the background polynomial.
            - The second dictionary contains the phase differences between the S-wave, D-wave, and P-wave amplitudes.
    """
    c_s0 = complex(c_s0_real, c_s0_imag)
    c_s1 = complex(c_s1_real, c_s1_imag)
    c_sm1 = complex(c_sm1_real, c_sm1_imag)
    c_d0 = complex(c_d0_real, c_d0_imag)
    c_p1 = complex(c_p1_real, 0.0) # P-wave is fixed to real to resolve unconstrained phase ambiguity

    s0_bw = c_s0 * vec_bw(mass, mass_b1, width_b1, 0)
    s1_bw = c_s1 * vec_bw(mass, mass_b1, width_b1, 0)
    sm1_bw = c_sm1 * vec_bw(mass, mass_b1, width_b1, 0)
    d0_bw = c_d0 * vec_bw(mass, mass_b1, width_b1, 2)
    p1_bw = c_p1 * vec_bw(mass, mass_rho_prime, width_rho_prime, 1)

    poly_bkg = bkg_polynomial(mass, a0, a1, a2, a3, a4)

    intensity_s0 = np.abs(s0_bw)**2
    intensity_s1 = np.abs(s1_bw)**2
    intensity_sm1 = np.abs(sm1_bw)**2
    intensity_d0 = np.abs(d0_bw)**2
    intensity_p1 = np.abs(p1_bw)**2

    phase_s0_s1 = bw_phase_difference(s0_bw, s1_bw)
    phase_s0_sm1 = bw_phase_difference(s0_bw, sm1_bw)
    phase_s0_d0 = bw_phase_difference(s0_bw, d0_bw)
    phase_s0_p1 = bw_phase_difference(s0_bw, p1_bw)
    phase_s1_p1 = bw_phase_difference(s1_bw, p1_bw)
    phase_s1_sm1 = bw_phase_difference(s1_bw, sm1_bw)
    phase_s1_d0 = bw_phase_difference(s1_bw, d0_bw)
    phase_sm1_p1 = bw_phase_difference(sm1_bw, p1_bw)
    phase_sm1_d0 = bw_phase_difference(sm1_bw, d0_bw)
    phase_d0_p1 = bw_phase_difference(d0_bw, p1_bw)

    intensities = {
        "intensity_s0": intensity_s0,
        "intensity_s1": intensity_s1,
        "intensity_sm1": intensity_sm1,
        "intensity_d0": intensity_d0,
        "intensity_p1": intensity_p1,
        "intensity_bkg": poly_bkg
    }

    phases = {
        "phase_s0_s1": phase_s0_s1,
        "phase_s0_sm1": phase_s0_sm1,
        "phase_s0_d0": phase_s0_d0,
        "phase_s0_p1": phase_s0_p1,
        "phase_s1_p1": phase_s1_p1,
        "phase_s1_sm1": phase_s1_sm1,
        "phase_s1_d0": phase_s1_d0,
        "phase_sm1_p1": phase_sm1_p1,
        "phase_sm1_d0": phase_sm1_d0,
        "phase_d0_p1": phase_d0_p1,
    }

    return (intensities, phases)
    


In [ ]:
def b1_rho_prime_bkg_residuals(
    params,
    mass: np.ndarray,
    intensities: dict[str, tuple[np.ndarray, np.ndarray]],
    phases: dict[str, tuple[np.ndarray, np.ndarray]],
) -> np.ndarray:
    """
    Calculate the residuals between the model and the data for both intensities and phases.

    Args:
        params: The parameters for the model.
        mass: The mass values to evaluate the model at.
        intensities: The observed intensities and their uncertainties for each wave.
        phases: The observed phase differences and their uncertainties for each wave pair.
    """
    predicted_intensities, predicted_phases = b1_rho_prime_bkg(
        mass,
        *params
    )

    # Calculate the residuals for intensities
    intensity_residuals = []
    for key, (i_obs, i_err) in intensities.items():
        if key == "intensity_bkg":
            continue # skip the background polynomial for residuals, since we have no direct observed data for it

        i_pred = predicted_intensities[key]
        i_bkg = predicted_intensities["intensity_bkg"]
        intensity_residuals.append(((i_pred+i_bkg) - i_obs) / i_err)

    # Calculate the residuals for phases
    phase_residuals = []
    for key, (p_obs, p_err) in phases.items():
        p_pred = predicted_phases[key]
        r_plus = vec_circular_residual(p_pred, p_obs, in_degrees=True) / p_err
        r_minus = vec_circular_residual(p_pred, -p_obs, in_degrees=True) / p_err
        r = np.where(np.abs(r_plus) < np.abs(r_minus), r_plus, r_minus)

        phase_residuals.append(r)

    # Combine the residuals
    return np.concatenate(intensity_residuals + phase_residuals)

In [ ]:
# Store the results of the fits for each t-bin
b1_rho_prime_bkg_fit_results = {}

In [ ]:
from scipy.optimize import least_squares
from matplotlib.ticker import ScalarFormatter, MaxNLocator

colors = plt.colormaps["Dark2"].colors # type: ignore

for bin_idx, (t_bin, results) in enumerate(binned_results.items()):

    # Get the amplitude and phase data values
    s0_name, s1_name, sm1_name, p1_name, d0_name = "p1p0S", "p1ppS", "p1pmS", "p1mpP", "p1p0D"
    s0_s1_name = results.phase_difference_dict[(s0_name, s1_name)]
    s0_d0_name = results.phase_difference_dict[(s0_name, d0_name)]
    s0_p1_name = results.phase_difference_dict[(s0_name, p1_name)]
    s0_sm1_name = results.phase_difference_dict[(s0_name, sm1_name)]
    s1_p1_name = results.phase_difference_dict[(s1_name, p1_name)]
    s1_d0_name = results.phase_difference_dict[(s1_name, d0_name)]
    s1_sm1_name = results.phase_difference_dict[(s1_name, sm1_name)]
    sm1_p1_name = results.phase_difference_dict[(sm1_name, p1_name)]
    sm1_d0_name = results.phase_difference_dict[(sm1_name, d0_name)]
    d0_p1_name = results.phase_difference_dict[(d0_name, p1_name)]

    intensity_s0 = results.fit_df[s0_name].to_numpy()
    intensity_s1 = results.fit_df[s1_name].to_numpy()
    intensity_sm1 = results.fit_df[sm1_name].to_numpy()
    intensity_p1 = results.fit_df[p1_name].to_numpy()
    intensity_d0 = results.fit_df[d0_name].to_numpy()

    intensity_s0_err = results.plot.intensity.get_bootstrap_error(s0_name).to_numpy()
    intensity_s1_err = results.plot.intensity.get_bootstrap_error(s1_name).to_numpy()
    intensity_sm1_err = results.plot.intensity.get_bootstrap_error(sm1_name).to_numpy()
    intensity_p1_err = results.plot.intensity.get_bootstrap_error(p1_name).to_numpy()
    intensity_d0_err = results.plot.intensity.get_bootstrap_error(d0_name).to_numpy()

    phase_s0_s1 = results.fit_df[s0_s1_name].to_numpy()
    phase_s0_d0 = results.fit_df[s0_d0_name].to_numpy()
    phase_s0_p1 = results.fit_df[s0_p1_name].to_numpy()
    phase_s0_sm1 = results.fit_df[s0_sm1_name].to_numpy()
    phase_s1_p1 = results.fit_df[s1_p1_name].to_numpy()
    phase_s1_d0 = results.fit_df[s1_d0_name].to_numpy()
    phase_s1_sm1 = results.fit_df[s1_sm1_name].to_numpy()
    phase_sm1_p1 = results.fit_df[sm1_p1_name].to_numpy()
    phase_sm1_d0 = results.fit_df[sm1_d0_name].to_numpy()
    phase_d0_p1 = results.fit_df[d0_p1_name].to_numpy()

    phase_s0_s1_err = results.plot.phase.get_bootstrap_error(s0_s1_name).to_numpy()
    phase_s0_d0_err = results.plot.phase.get_bootstrap_error(s0_d0_name).to_numpy()
    phase_s0_p1_err = results.plot.phase.get_bootstrap_error(s0_p1_name).to_numpy()
    phase_s0_sm1_err = results.plot.phase.get_bootstrap_error(s0_sm1_name).to_numpy()
    phase_s1_p1_err = results.plot.phase.get_bootstrap_error(s1_p1_name).to_numpy()
    phase_s1_d0_err = results.plot.phase.get_bootstrap_error(s1_d0_name).to_numpy()
    phase_s1_sm1_err = results.plot.phase.get_bootstrap_error(s1_sm1_name).to_numpy()
    phase_sm1_p1_err = results.plot.phase.get_bootstrap_error(sm1_p1_name).to_numpy()
    phase_sm1_d0_err = results.plot.phase.get_bootstrap_error(sm1_d0_name).to_numpy()
    phase_d0_p1_err = results.plot.phase.get_bootstrap_error(d0_p1_name).to_numpy()

    data_masses = np.array(results.get_mass_centers())
    bin_width = results.get_mass_bin_width()


    # --- Initial guesses for the fit parameters ---

    # set the magnitude of the complex coefficients to the square root
    # of the mean intensity for each wave, and set the phase to 0.
    c_s0_mag = np.sqrt(np.mean(intensity_s0))
    c_s1_mag = np.sqrt(np.mean(intensity_s1))
    c_sm1_mag = np.sqrt(np.mean(intensity_sm1))
    c_d0_mag = np.sqrt(np.mean(intensity_d0))
    c_p1_mag = np.sqrt(np.mean(intensity_p1))
    
    c_s0_guess = c_s0_mag * np.exp(1j * 0)
    c_s1_guess = c_s1_mag * np.exp(1j * 0)
    c_sm1_guess = c_sm1_mag * np.exp(1j * 0)
    c_d0_guess = c_d0_mag * np.exp(1j * 0)
    c_p1_guess = c_p1_mag * np.exp(1j * 0)

    # use PDG values for the b1 and rho' masses and widths as initial guesses
    mass_b1_guess = 1.235
    mass_rho_prime_guess = 1.465
    width_b1_guess = 0.142
    width_rho_prime_guess = 0.400

    # assume background is zero
    a0_guess, a1_guess, a2_guess, a3_guess, a4_guess = 0.0, 0.0, 0.0, 0.0, 0.0

    popt = np.array([
        c_s0_guess.real,
        c_s0_guess.imag,
        c_s1_guess.real,
        c_s1_guess.imag,
        c_sm1_guess.real,
        c_sm1_guess.imag,
        c_d0_guess.real,
        c_d0_guess.imag,
        mass_b1_guess,
        width_b1_guess,
        c_p1_guess.real,
        mass_rho_prime_guess,
        width_rho_prime_guess,
        a0_guess,
        a1_guess,
        a2_guess,
        a3_guess,
        a4_guess
    ], dtype=float)

    fit = least_squares(
        b1_rho_prime_bkg_residuals,
        x0=popt,
        args=(
            data_masses,
            {
                "intensity_s0": (intensity_s0, intensity_s0_err),
                "intensity_s1": (intensity_s1, intensity_s1_err),
                "intensity_sm1": (intensity_sm1, intensity_sm1_err),
                "intensity_d0": (intensity_d0, intensity_d0_err),
                "intensity_p1": (intensity_p1, intensity_p1_err),
            },
            {
                "phase_s0_s1": (phase_s0_s1, phase_s0_s1_err),
                "phase_s0_d0": (phase_s0_d0, phase_s0_d0_err),
                "phase_s0_p1": (phase_s0_p1, phase_s0_p1_err),
                "phase_s0_sm1": (phase_s0_sm1, phase_s0_sm1_err),
                "phase_s1_p1": (phase_s1_p1, phase_s1_p1_err),
                "phase_s1_d0": (phase_s1_d0, phase_s1_d0_err),
                "phase_s1_sm1": (phase_s1_sm1, phase_s1_sm1_err),
                "phase_sm1_p1": (phase_sm1_p1, phase_sm1_p1_err),
                "phase_sm1_d0": (phase_sm1_d0, phase_sm1_d0_err),
                "phase_d0_p1": (phase_d0_p1, phase_d0_p1_err),
            },
        ),
        bounds=(
            [
                -np.inf, -np.inf, -np.inf, -np.inf, -np.inf, -np.inf, -np.inf, -np.inf, 
                1.2, 0.1, -np.inf, 1.2, 0.2,
                -np.inf, -np.inf, -np.inf, -np.inf, -np.inf
            ],
            [
                np.inf, np.inf, np.inf, np.inf, np.inf, np.inf, np.inf, np.inf,
                1.4, 0.4, np.inf, 1.7, 0.7,
                np.inf, np.inf, np.inf, np.inf, np.inf
            ]
        )
    )

    # Extract the fit parameters and their uncertainties
    popt = fit.x
    mass_b1_fit, width_b1_fit = popt[8], popt[9]
    mass_rho_prime_fit, width_rho_prime_fit = popt[11], popt[12]
    a0_fit, a1_fit, a2_fit, a3_fit, a4_fit = popt[13:]

    J = fit.jac
    cov = np.linalg.inv(J.T @ J)
    dof = len(fit.fun) - len(popt)
    s_sq = 2 * fit.cost / dof
    cov = cov * s_sq
    perr = np.sqrt(np.diag(cov))

    mass_b1_fit_err, width_b1_fit_err = perr[8], perr[9]
    mass_rho_prime_fit_err, width_rho_prime_fit_err = perr[11], perr[12]
    a0_fit_err, a1_fit_err, a2_fit_err, a3_fit_err, a4_fit_err = perr[13:]

    b1_rho_prime_bkg_fit_results[t_bin] = {
        "mass_b1": mass_b1_fit,
        "width_b1": width_b1_fit,
        "mass_rho_prime": mass_rho_prime_fit,
        "width_rho_prime": width_rho_prime_fit,
        "mass_b1_err": mass_b1_fit_err,
        "width_b1_err": width_b1_fit_err,
        "mass_rho_prime_err": mass_rho_prime_fit_err,
        "width_rho_prime_err": width_rho_prime_fit_err,
    }


    # Create a 4x4 grid of subplots for the amplitudes and phases
    fig, axs = plt.subplots(
        5, 5,
        layout="constrained",
        dpi=300
    )

    grid_amplitudes = [
        [sm1_name, s0_sm1_name, s1_sm1_name, sm1_p1_name, sm1_d0_name],
        [s0_name, s0_s1_name, s0_p1_name, s0_d0_name],
        [s1_name, s1_p1_name, s1_d0_name],
        [p1_name, d0_p1_name],
        [d0_name]
    ]

    translate_dict = {
        s0_name: "intensity_s0",
        s1_name: "intensity_s1",
        sm1_name: "intensity_sm1",
        p1_name: "intensity_p1",
        d0_name: "intensity_d0",
        s0_s1_name: "phase_s0_s1",
        s0_d0_name: "phase_s0_d0",
        s0_p1_name: "phase_s0_p1",
        s0_sm1_name: "phase_s0_sm1",
        s1_p1_name: "phase_s1_p1",
        s1_d0_name: "phase_s1_d0",
        s1_sm1_name: "phase_s1_sm1",
        sm1_p1_name: "phase_sm1_p1",
        sm1_d0_name: "phase_sm1_d0",
        d0_p1_name: "phase_d0_p1"
    }

    fine_masses = np.linspace(1.0, 1.8, 1000)
    fit_intensities, fit_phases = b1_rho_prime_bkg(fine_masses, *popt)

    for row in range(5):
        for col in range(5):
            if col < row: # skip lower triangle, redundant information
                axs[row, col].axis('off')
                continue
        
            amp_name = grid_amplitudes[row][col-row]
            y_values = results.fit_df[amp_name].to_numpy()
            y_errors = results.plot.intensity.get_bootstrap_error(amp_name).to_numpy()

            if row == col:
                # ==== PLOT INTENSITIES ====                

                # plot original data with error bars
                color = colors[2] if "S" in amp_name or "D" in amp_name else colors[3]                
                axs[row, col].errorbar(
                    x=data_masses,
                    xerr=bin_width/2,
                    y=y_values,
                    yerr=y_errors,
                    color=color,
                    marker=".",
                    markersize=4,
                    linestyle="",
                    alpha=0.7
                )
                
                # plot the fitted model, but add the background polynomial to the predicted intensities
                fit_intensity_result = fit_intensities[translate_dict[amp_name]] + fit_intensities["intensity_bkg"]
                axs[row, col].plot(
                    fine_masses,
                    fit_intensity_result,
                    color=color,
                    marker="",
                    linestyle="-",
                    linewidth=1.5,                    
                )

                # plot the background polynomial as a dashed line
                axs[row, col].plot(
                    fine_masses,
                    fit_intensities["intensity_bkg"],
                    color=colors[0],
                    marker="",
                    linestyle="--",
                    linewidth=1.5,
                    alpha=0.7,
                )

                axs[row, col].set_ylim(bottom=0)
                axs[row, col].yaxis.set_major_locator(MaxNLocator(nbins=5, integer=True))
                axs[row, col].ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
                axs[row, col].yaxis.set_major_formatter(ScalarFormatter(useMathText=True))
                axs[row, col].yaxis.get_major_formatter().set_powerlimits((0, 0))
                axs[row, col].yaxis.set_offset_position("left")
                axs[row, col].yaxis.get_offset_text().set_fontsize(8)
                axs[row, col].tick_params(axis="y", labelsize=8)
                axs[row, col].set_xlabel(r"$\omega\pi^0$ inv. mass (GeV)", fontsize=8)
                axs[row, col].set_ylabel(f"Events / {bin_width:.3f} GeV", loc="top", fontsize=8)


            else:
                # ==== PLOT PHASES ====

                # plot original data with error bars
                axs[row,col].errorbar(
                    x=data_masses,
                    xerr=bin_width/2,
                    y=y_values,
                    yerr=y_errors,
                    color="black",
                    marker=".",
                    markersize=4,
                    linestyle="",
                    alpha=0.7
                )
                axs[row,col].errorbar(
                    x=data_masses,
                    xerr=bin_width/2,
                    y=-y_values, # sign ambiguous phase 
                    yerr=y_errors,
                    color="black",
                    marker=".",
                    markersize=4,
                    linestyle="",
                    alpha=0.7
                )
                axs[row,col].set_ylim(-180, 180)
                axs[row,col].set_yticks(np.linspace(-180, 180, 5))
                axs[row,col].set_yticklabels([f"{x:.0f}" for x in np.linspace(-180, 180, 5)], fontsize=8)

                # plot the fitted model. No polynomial background interference is present
                axs[row, col].plot(
                    fine_masses,
                    fit_phases[translate_dict[amp_name]],
                    color="black",
                    marker="",
                    linestyle="-",
                    linewidth=1.5,                    
                )

            
            axs[row, col].set_xticks(np.linspace(1.0, 1.8, 6))
            axs[row, col].set_xticklabels([f"{x:.1f}" for x in np.linspace(1.0, 1.8, 6)], fontsize=8)
            latex_amp_name = utils.convert_amp_name(amp_name)
            axs[row, col].set_title(latex_amp_name, loc="right", fontsize=10)

            

    t_low, t_high = results.get_t_edges()
    fig.suptitle(f"{t_low:.2f} < t < {t_high:.2f} GeV^2", ha="center", fontsize=16)

    ds0_ratio = np.sqrt(fit_intensities["intensity_d0"] / fit_intensities["intensity_s0"])
    e852_mass_range = [1.155, 1.315]
    filtered_ds0_ratio = ds0_ratio[(fine_masses >= e852_mass_range[0]) & (fine_masses <= e852_mass_range[1])]
    mean_ds0_ratio = np.mean(filtered_ds0_ratio)
    

    fig.text(0.1, 0.05, 
        (
            "BW Fit Results:"
            "\n"
            rf"$M_{{b1}} = {mass_b1_fit:.3f} \pm {mass_b1_fit_err:.3f}$ GeV"
            "\n"
            rf"$\Gamma_{{b1}} = {width_b1_fit:.3f} \pm {width_b1_fit_err:.3f}$ GeV"
            "\n\n"
            rf"$M_{{\rho'}} = {mass_rho_prime_fit:.3f} \pm {mass_rho_prime_fit_err:.3f}$ GeV"
            "\n"
            rf"$\Gamma_{{\rho'}} = {width_rho_prime_fit:.3f} \pm {width_rho_prime_fit_err:.3f}$ GeV"
            "\n\n"
            rf"$\overline{{D/S}}_0 = {mean_ds0_ratio:.3f} \pm {np.std(filtered_ds0_ratio):.3f}$"
            "\n\n"
            "Background Polynomial Coefficients:"
            "\n"
            r"$a_0 + a_1 m + a_2 m^2 + a_3 m^3 + a_4 m^4$"
            "\n"
            rf"$a_0$ = {a0_fit:.1e} $\pm$ {a0_fit_err:.1e}"
            "\n"
            rf"$a_1$ = {a1_fit:.1e} $\pm$ {a1_fit_err:.1e}"
            "\n"
            rf"$a_2$ = {a2_fit:.1e} $\pm$ {a2_fit_err:.1e}"
            "\n"
            rf"$a_3$ = {a3_fit:.1e} $\pm$ {a3_fit_err:.1e}"
            "\n"
            rf"$a_4$ = {a4_fit:.1e} $\pm$ {a4_fit_err:.1e}"
        ),
        bbox=dict(facecolor='white', alpha=0.5, edgecolor='black'),
        ha="center", fontsize=14
    )

    plt.show()



## Plot Resonance parameters

This will likely get expanded further, but I want to simply see how the b1 and rho' parameters differ across the models and -t bins

In [ ]:
t_avg = []
b1_masses = [[], []]
b1_masses_errs = [[], []]
b1_widths = [[], []]
b1_widths_errs = [[], []]
rho_prime_masses = [[], []]
rho_prime_masses_errs = [[], []]
rho_prime_widths = [[], []]
rho_prime_widths_errs = [[], []]

for bin_idx, (t_bin, results) in enumerate(binned_results.items()):

    t_low, t_high = results.get_t_edges()
    t_avg.append((t_low + t_high) / 2)

    model_results = b1_rho_prime_fit_results[t_bin]
    b1_masses[0].append(model_results['mass_b1'])
    b1_masses_errs[0].append(model_results['mass_b1_err'])
    b1_widths[0].append(model_results['width_b1'])
    b1_widths_errs[0].append(model_results['width_b1_err'])
    rho_prime_masses[0].append(model_results['mass_rho_prime'])
    rho_prime_masses_errs[0].append(model_results['mass_rho_prime_err'])
    rho_prime_widths[0].append(model_results['width_rho_prime'])
    rho_prime_widths_errs[0].append(model_results['width_rho_prime_err'])

    model_results = b1_rho_prime_bkg_fit_results[t_bin]
    b1_masses[1].append(model_results['mass_b1'])
    b1_masses_errs[1].append(model_results['mass_b1_err'])
    b1_widths[1].append(model_results['width_b1'])
    b1_widths_errs[1].append(model_results['width_b1_err'])
    rho_prime_masses[1].append(model_results['mass_rho_prime'])
    rho_prime_masses_errs[1].append(model_results['mass_rho_prime_err'])
    rho_prime_widths[1].append(model_results['width_rho_prime'])
    rho_prime_widths_errs[1].append(model_results['width_rho_prime_err'])



fig, axs = plt.subplots(
    1, 4,
    figsize=(10, 6),
    sharey=True,
    layout="constrained",
    dpi=300
)

pdg_b1_mass = 1.2295
pdg_b1_mass_err = 0.0032
pdg_b1_width = 0.142
pdg_b1_width_err = 0.009
pdg_rho_prime_mass = 1.465
pdg_rho_prime_mass_err = 0.025
pdg_rho_prime_width = 0.400
pdg_rho_prime_width_err = 0.060

# b1 masses
axs[0].errorbar(
    b1_masses[0], t_avg, xerr=b1_masses_errs[0], marker=".", linestyle="", color=colors[2], label="No Background", alpha=0.7
)
axs[0].errorbar(
    b1_masses[1], t_avg, xerr=b1_masses_errs[1], marker="d", linestyle="", color=colors[2], label="With Background", alpha=0.7
)
axs[0].axvline(
    pdg_b1_mass, color="black", linestyle="--", label="PDG Value"
)
axs[0].fill_betweenx(
    y=[min(t_avg), max(t_avg)],
    x1=pdg_b1_mass - pdg_b1_mass_err,
    x2=pdg_b1_mass + pdg_b1_mass_err,
    color="black",
    alpha=0.2,
    label="PDG Uncertainty"
)
axs[0].set_ylabel(r"$-t$ (GeV$^2$)", fontsize=12)
axs[0].set_xlabel(r"$M_{b_1}$ (GeV)", fontsize=12)

# b1 widths
axs[1].errorbar(
    b1_widths[0], t_avg, xerr=b1_widths_errs[0], marker=".", linestyle="", color=colors[2], label="No Background", alpha=0.7
)
axs[1].errorbar(
    b1_widths[1], t_avg, xerr=b1_widths_errs[1], marker="d", linestyle="", color=colors[2], label="With Background", alpha=0.7
)
axs[1].axvline(
    pdg_b1_width, color="black", linestyle="--", label="PDG Value"
)
axs[1].fill_betweenx(
    y=[min(t_avg), max(t_avg)],
    x1=pdg_b1_width - pdg_b1_width_err,
    x2=pdg_b1_width + pdg_b1_width_err,
    color="black",
    alpha=0.2,
    label="PDG Uncertainty"
)
axs[1].set_xlabel(r"$\Gamma_{b_1}$ (GeV)", fontsize=12)

# rho' masses
axs[2].errorbar(
    rho_prime_masses[0], t_avg, xerr=rho_prime_masses_errs[0], marker=".", linestyle="", color=colors[3], label="No Background", alpha=0.7
)
axs[2].errorbar(
    rho_prime_masses[1], t_avg, xerr=rho_prime_masses_errs[1], marker="d", linestyle="", color=colors[3], label="With Background", alpha=0.7
)
axs[2].axvline(
    pdg_rho_prime_mass, color="black", linestyle="--", label="PDG Value"
)
axs[2].fill_betweenx(
    y=[min(t_avg), max(t_avg)],
    x1=pdg_rho_prime_mass - pdg_rho_prime_mass_err,
    x2=pdg_rho_prime_mass + pdg_rho_prime_mass_err,
    color="black",
    alpha=0.2,
    label="PDG Uncertainty"
)
axs[2].set_xlabel(r"$M_{\rho'}$ (GeV)", fontsize=12)


# rho' widths
axs[3].errorbar(
    rho_prime_widths[0], t_avg, xerr=rho_prime_widths_errs[0], marker=".", linestyle="", color=colors[3], label="No Background", alpha=0.7
)
axs[3].errorbar(
    rho_prime_widths[1], t_avg, xerr=rho_prime_widths_errs[1], marker="d", linestyle="", color=colors[3], label="With Background", alpha=0.7
)
axs[3].axvline(
    pdg_rho_prime_width, color="black", linestyle="--", label="PDG Value"
)
axs[3].fill_betweenx(
    y=[min(t_avg), max(t_avg)],
    x1=pdg_rho_prime_width - pdg_rho_prime_width_err,
    x2=pdg_rho_prime_width + pdg_rho_prime_width_err,
    color="black",
    alpha=0.2,
    label="PDG Uncertainty"
)
axs[3].set_xlabel(r"$\Gamma_{\rho'}$ (GeV)", fontsize=12)

# fig legend with dummy lines / markers to indicate common style
handles, labels = axs[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.75))

plt.show()
